In [ ]:
!pip install megadetector

In [ ]:
from megadetector.utils import url_utils
from megadetector.visualization import visualization_utils as vis_utils
from megadetector.detection import run_detector

image_url = 'https://github.com/agentmorris/MegaDetector/raw/main/images/orinoquia-thumb-web.jpg'
temporary_filename = url_utils.download_url(image_url)
image = vis_utils.load_image(temporary_filename)

model = run_detector.load_detector('MDV5A')
result = model.generate_detections_one_image(image)

detections_above_threshold = [d for d in result['detections'] if d['conf'] > 0.2]
print('Found {} detections above threshold'.format(len(detections_above_threshold)))

In [ ]:
import pandas as pd
import urllib.request
import zipfile

# Download just the metadata (small - a few MB, not the 176GB image archive)
url = "https://storage.googleapis.com/public-datasets-lila/wellingtoncameratraps/wellington_camera_traps.csv.zip"
urllib.request.urlretrieve(url, "wellington_camera_traps.csv.zip")

# Unzip it
with zipfile.ZipFile("wellington_camera_traps.csv.zip", "r") as z:
    z.extractall(".")

# Load it and take a look
df = pd.read_csv("wellington_camera_traps.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
import json

json_url = "https://storage.googleapis.com/public-datasets-lila/wellingtoncameratraps/wellington_camera_traps.json.zip"
urllib.request.urlretrieve(json_url, "wellington_camera_traps.json.zip")

with zipfile.ZipFile("wellington_camera_traps.json.zip", "r") as z:
    z.extractall(".")

with open("wellington_camera_traps.json") as f:
    coco_data = json.load(f)

print(coco_data.keys())
print(len(coco_data['images']))
print(coco_data['images'][0])

In [ ]:
from PIL import Image

base_url = "https://storage.googleapis.com/public-datasets-lila/wellington-unzipped/images/"

test_entry = coco_data['images'][0]
test_url = base_url + test_entry['file_name']
print(test_url)

urllib.request.urlretrieve(test_url, "test_image.jpg")
img = Image.open("test_image.jpg")
display(img)

print(coco_data['annotations'][0])
print(coco_data['categories'][:5])

In [ ]:
import random

# Build lookup dictionaries: ID -> actual value
image_id_to_filename = {img['id']: img['file_name'] for img in coco_data['images']}
category_id_to_name = {cat['id']: cat['name'] for cat in coco_data['categories']}

# Walk through every annotation, and use the two dictionaries above
# to turn its IDs into an actual (filename, label) pair
all_pairs = []
for ann in coco_data['annotations']:
    filename = image_id_to_filename.get(ann['image_id'])
    label = category_id_to_name.get(ann['category_id'])
    if filename and label:
        all_pairs.append((filename, label))

print(f"Total labeled images: {len(all_pairs)}")

# Randomly sample our working subset
random.seed(42)  # fixes the randomness so the same 200 get picked every time we rerun this
sample_size = 200
sample = random.sample(all_pairs, sample_size)

print(f"Sampled {len(sample)} images")
print(sample[:5])

In [ ]:
import os

output_dir = "sample_images"
os.makedirs(output_dir, exist_ok=True)

downloaded = []
failed = []

for i, (filename, label) in enumerate(sample):
    url = base_url + filename
    local_path = os.path.join(output_dir, filename)
    try:
        urllib.request.urlretrieve(url, local_path)
        downloaded.append((filename, label))
    except Exception as e:
        failed.append((filename, str(e)))

    if (i + 1) % 25 == 0:
        print(f"Downloaded {i + 1}/{len(sample)}...")

print(f"\nDone. {len(downloaded)} succeeded, {len(failed)} failed.")
if failed:
    print("Failed files:", failed[:5])

In [ ]:
import csv

with open("sample_ground_truth.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "true_label"])
    writer.writerows(downloaded)

print("Saved ground truth reference to sample_ground_truth.csv")

In [ ]:
from megadetector.detection.run_detector_batch import load_and_run_detector_batch, write_results_to_file

image_folder = 'sample_images'
output_file = 'detection_results.json'
model_name = 'MDV5A'

results = load_and_run_detector_batch(model_name, image_folder)

write_results_to_file(results, output_file, relative_path_base=image_folder, detector_file=model_name)

print(f"Processed {len(results['images'])} images")
print(results['images'][0])

In [ ]:
import json

with open("detection_results.json") as f:
    detection_results = json.load(f)

print(detection_results.keys())
print(len(detection_results['images']))
print(detection_results['images'][0])

In [ ]:
print(detection_results['detection_categories'])

CONF_THRESHOLD = 0.2
has_detection = 0
blank = 0

for img in detection_results['images']:
    confident_dets = [d for d in img['detections'] if float(d['conf']) >= CONF_THRESHOLD]
    if confident_dets:
        has_detection += 1
    else:
        blank += 1

print(f"Images with a confident detection: {has_detection}")
print(f"Images filtered out as blank: {blank}")
print(f"That's {blank/200*100:.1f}% filtered")

In [ ]:
truth_lookup = dict(downloaded)  # filename -> true label

true_positives = 0   # MD says animal, truth agrees
true_negatives = 0   # MD says blank, truth agrees (correctly filtered)
false_positives = 0  # MD says animal, but truth says empty (kept unnecessarily)
false_negatives = 0  # MD says blank, but truth says there WAS an animal (wrongly discarded)

for img in detection_results['images']:
    true_label = truth_lookup.get(img['file'])
    if true_label is None:
        continue

    confident_dets = [d for d in img['detections'] if float(d['conf']) >= CONF_THRESHOLD]
    md_says_animal = len(confident_dets) > 0
    truth_says_animal = (true_label != 'empty')

    if md_says_animal and truth_says_animal:
        true_positives += 1
    elif not md_says_animal and not truth_says_animal:
        true_negatives += 1
    elif md_says_animal and not truth_says_animal:
        false_positives += 1
    else:
        false_negatives += 1

print(f"Correctly kept (animal, kept): {true_positives}")
print(f"Correctly filtered (blank, filtered): {true_negatives}")
print(f"Wrongly kept (blank, but kept): {false_positives}")
print(f"Wrongly filtered (animal, but discarded): {false_negatives}")

In [ ]:
from PIL import Image

false_negative_examples = []
for img in detection_results['images']:
    true_label = truth_lookup.get(img['file'])
    if true_label is None or true_label == 'empty':
        continue
    confident_dets = [d for d in img['detections'] if float(d['conf']) >= CONF_THRESHOLD]
    if len(confident_dets) == 0:
        false_negative_examples.append((img['file'], true_label))

print(false_negative_examples)

for filename, label in false_negative_examples[:3]:
    print(f"{filename} — labeled: {label}")
    display(Image.open(f"sample_images/{filename}"))

In [ ]:
!pip install anthropic

In [ ]:
import anthropic
from google.colab import userdata

api_key = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=api_key)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ]
)

print(response.content[0].text)

In [ ]:
from PIL import Image

def crop_detection(image_path, bbox):
    img = Image.open(image_path)
    width, height = img.size

    x, y, w, h = bbox
    left = x * width
    top = y * height
    right = (x + w) * width
    bottom = (y + h) * height

    return img.crop((left, top, right, bottom))

test_img = detection_results['images'][0]
confident_dets = [d for d in test_img['detections'] if float(d['conf']) >= CONF_THRESHOLD]

if confident_dets:
    bbox = confident_dets[0]['bbox']
    image_path = f"sample_images/{test_img['file']}"
    original = Image.open(image_path)
    cropped = crop_detection(image_path, bbox)
    display(cropped)
    print(f"Original: {original.size}, Cropped: {cropped.size}")
else:
    print("First image had no confident detection, we'll need to pick a different test one")

In [ ]:
def crop_detection(image_path, bbox, padding=0.15):
    img = Image.open(image_path)
    width, height = img.size

    x, y, w, h = bbox
    pad_w = w * padding
    pad_h = h * padding

    left = max(0, (x - pad_w) * width)
    top = max(0, (y - pad_h) * height)
    right = min(width, (x + w + pad_w) * width)
    bottom = min(height, (y + h + pad_h) * height)

    return img.crop((left, top, right, bottom))

In [ ]:
import base64
from io import BytesIO

def image_to_base64(pil_image):
    buffer = BytesIO()
    pil_image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

image_b64 = image_to_base64(cropped)

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_b64,
                    },
                },
                {
                    "type": "text",
                    "text": "This is a cropped camera-trap photo from New Zealand. What species is most likely shown? Give your best single guess and a one-sentence field-note-style description of what the animal appears to be doing. If you genuinely can't tell, say so."
                }
            ],
        }
    ],
)

print(response.content[0].text)
print(f"\nTrue label was: {test_img['file']}")

In [ ]:
category_names = [cat['name'] for cat in coco_data['categories']]
print(category_names)

In [ ]:
category_list_str = ", ".join(category_names)

prompt_text = f"""This is a cropped camera-trap photo from Wellington, New Zealand.

Choose the single most likely category from this exact list (use the exact spelling):
{category_list_str}

Respond in exactly this format:
CATEGORY: <one category from the list above>
NOTE: <one short sentence describing what you see>

If you genuinely cannot tell, use CATEGORY: unclassifiable."""

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=150,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64}},
                {"type": "text", "text": prompt_text}
            ],
        }
    ],
)

print(response.content[0].text)
print(f"\nTrue label was: {truth_lookup.get(test_img['file'])}")

In [ ]:
def parse_response(text):
    category = None
    note = None
    for line in text.split("\n"):
        if line.startswith("CATEGORY:"):
            category = line.replace("CATEGORY:", "").strip()
        elif line.startswith("NOTE:"):
            note = line.replace("NOTE:", "").strip()
    return category, note

# Test it on the response we just got
test_category, test_note = parse_response(response.content[0].text)
print(f"Parsed category: '{test_category}'")
print(f"Parsed note: '{test_note}'")
print(f"Matches valid category list: {test_category in category_names}")

In [ ]:
import time

confident_images = []
for img in detection_results['images']:
    dets = [d for d in img['detections'] if float(d['conf']) >= CONF_THRESHOLD]
    if dets:
        confident_images.append((img['file'], dets[0]['bbox']))

print(f"Processing {len(confident_images)} images...")

results_stage2 = []

for i, (filename, bbox) in enumerate(confident_images):
    image_path = f"sample_images/{filename}"
    try:
        cropped_img = crop_detection(image_path, bbox)
        img_b64 = image_to_base64(cropped_img)

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=150,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                        {"type": "text", "text": prompt_text}
                    ],
                }
            ],
        )

        category, note = parse_response(response.content[0].text)
        results_stage2.append({
            "filename": filename,
            "predicted": category,
            "note": note,
            "true_label": truth_lookup.get(filename)
        })

    except Exception as e:
        results_stage2.append({
            "filename": filename,
            "predicted": None,
            "note": f"ERROR: {e}",
            "true_label": truth_lookup.get(filename)
        })

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(confident_images)}...")

print(f"\nDone. {len(results_stage2)} results collected.")

In [ ]:
import json
with open("stage2_results.json", "w") as f:
    json.dump(results_stage2, f, indent=2)
print("Saved.")

In [ ]:
total = len(results_stage2)
correct = 0
abstained = 0
wrong = 0

for r in results_stage2:
    if r['predicted'] == r['true_label']:
        correct += 1
    elif r['predicted'] == 'unclassifiable':
        abstained += 1
    else:
        wrong += 1

print(f"Total: {total}")
print(f"Correct: {correct} ({correct/total*100:.1f}%)")
print(f"Abstained (said unclassifiable): {abstained} ({abstained/total*100:.1f}%)")
print(f"Wrong: {wrong} ({wrong/total*100:.1f}%)")
print(f"\nAccuracy when Claude was willing to commit: {correct/(correct+wrong)*100:.1f}%")

In [ ]:
for r in results_stage2[:15]:
    print(f"predicted: {r['predicted']!r}  |  true: {r['true_label']!r}")

unique_predicted = set(r['predicted'] for r in results_stage2)
print("\nUnique predicted values Claude actually returned:")
print(unique_predicted)

In [ ]:
from collections import Counter

predicted_counts = Counter(r['predicted'] for r in results_stage2)
true_counts = Counter(r['true_label'] for r in results_stage2)

print("What Claude predicted, and how often:")
for label, count in predicted_counts.most_common():
    print(f"  {label}: {count}")

print("\nWhat the true labels actually were:")
for label, count in true_counts.most_common():
    print(f"  {label}: {count}")

In [ ]:
confident_bbox_lookup = dict(confident_images)  # filename -> bbox, still in memory from the batch run

possum_bird_confusions = [r for r in results_stage2 if r['predicted'] == 'possum' and r['true_label'] == 'bird']
print(f"Found {len(possum_bird_confusions)} cases: Claude said possum, true label was bird")

for r in possum_bird_confusions[:3]:
    print(f"\n{r['filename']}")
    print(f"Claude's note: {r['note']}")
    image_path = f"sample_images/{r['filename']}"
    bbox = confident_bbox_lookup[r['filename']]
    cropped_img = crop_detection(image_path, bbox)
    display(cropped_img)

In [ ]:
total = len(results_stage2)
correct = sum(1 for r in results_stage2 if r['predicted'] == r['true_label'])
abstained = sum(1 for r in results_stage2 if r['predicted'] == 'unclassifiable')
wrong = total - correct - abstained
print(f"Correct: {correct} ({correct/total*100:.1f}%)")
print(f"Abstained: {abstained} ({abstained/total*100:.1f}%)")
print(f"Wrong: {wrong} ({wrong/total*100:.1f}%)")

In [ ]:
prompt_text_v2 = f"""This is a cropped photo from a wildlife camera trap.

Look carefully at the actual visual details: body shape and size, fur or feather texture, limb shape, any visible head, eye, ear, or beak features.

Choose the single most likely category from this exact list (use the exact spelling):
{category_list_str}

Respond in exactly this format:
OBSERVATION: <2-3 objective visual details you actually see in the image>
CATEGORY: <one category from the list above>
NOTE: <one short sentence>

Base your answer only on what is visible in the image, not assumptions about what species are common in any particular region. If you genuinely cannot tell, use CATEGORY: unclassifiable."""

In [ ]:
for r in possum_bird_confusions[:3]:
    image_path = f"sample_images/{r['filename']}"
    bbox = confident_bbox_lookup[r['filename']]
    cropped_img = crop_detection(image_path, bbox)
    img_b64 = image_to_base64(cropped_img)

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                {"type": "text", "text": prompt_text_v2}
            ],
        }],
    )
    print(f"\n{r['filename']} (true label: bird)")
    print(response.content[0].text)

In [ ]:
results_stage2_v2 = []

for i, (filename, bbox) in enumerate(confident_images):
    image_path = f"sample_images/{filename}"
    try:
        cropped_img = crop_detection(image_path, bbox)
        img_b64 = image_to_base64(cropped_img)

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                    {"type": "text", "text": prompt_text_v2}
                ],
            }],
        )

        text = response.content[0].text
        category = None
        for line in text.split("\n"):
            if line.startswith("CATEGORY:"):
                category = line.replace("CATEGORY:", "").strip()

        results_stage2_v2.append({
            "filename": filename,
            "predicted": category,
            "true_label": truth_lookup.get(filename)
        })

    except Exception as e:
        results_stage2_v2.append({"filename": filename, "predicted": None, "true_label": truth_lookup.get(filename)})

    if (i + 1) % 20 == 0:
        print(f"Processed {i + 1}/{len(confident_images)}...")

print(f"\nDone. {len(results_stage2_v2)} results.")

In [ ]:
import json
with open("stage2_results_v2.json", "w") as f:
    json.dump(results_stage2_v2, f, indent=2)
print("Saved.")

In [ ]:
def score(results):
    total = len(results)
    correct = sum(1 for r in results if r['predicted'] == r['true_label'])
    abstained = sum(1 for r in results if r['predicted'] == 'unclassifiable')
    wrong = total - correct - abstained
    return correct, abstained, wrong, total

c1, a1, w1, t1 = score(results_stage2)
c2, a2, w2, t2 = score(results_stage2_v2)

print("BEFORE (with location hint):")
print(f"  Correct: {c1} ({c1/t1*100:.1f}%)  Abstained: {a1} ({a1/t1*100:.1f}%)  Wrong: {w1} ({w1/t1*100:.1f}%)")

print("\nAFTER (location hint removed):")
print(f"  Correct: {c2} ({c2/t2*100:.1f}%)  Abstained: {a2} ({a2/t2*100:.1f}%)  Wrong: {w2} ({w2/t2*100:.1f}%)")

In [ ]:
from collections import Counter
v2_predicted_counts = Counter(r['predicted'] for r in results_stage2_v2)
print("v2 predictions:")
for label, count in v2_predicted_counts.most_common():
    print(f"  {label}: {count}")

In [ ]:
import json, random, re
from collections import Counter
from PIL import Image, ImageOps, ImageDraw

def to_b64(pil_img):
    buf = BytesIO()
    pil_img.convert("RGB").save(buf, format="JPEG", quality=90)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def img_block(pil_img):
    return {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": to_b64(pil_img)}}

def parse_category(text):
    m = re.search(r"CATEGORY\s*:\s*\**\s*([a-z ]+)", text, re.IGNORECASE)
    if not m:
        return None
    cat = m.group(1).strip().lower().rstrip(".")
    return cat if cat in category_names else None

def score_results(results):
    t = len(results)
    c = sum(1 for r in results if r['predicted'] == r['true_label'])
    a = sum(1 for r in results if r['predicted'] == 'unclassifiable')
    return c, a, t - c - a, t

random.seed(7)
non_bird = [(f, b) for f, b in confident_images if truth_lookup.get(f) != 'bird']
birds    = [(f, b) for f, b in confident_images if truth_lookup.get(f) == 'bird']
dev_set  = non_bird + random.sample(birds, 25)
dev_files = {f for f, _ in dev_set}
print(f"Dev set: {len(dev_set)} images  ({len(non_bird)} non-bird + 25 bird)")

def run_experiment(name, build_content, model, subset=dev_set):
    results = []
    for i, (filename, bbox) in enumerate(subset):
        try:
            resp = client.messages.create(
                model=model, max_tokens=250,
                messages=[{"role": "user", "content": build_content(filename, bbox)}],
            )
            text = resp.content[0].text
            results.append({"filename": filename, "predicted": parse_category(text),
                            "true_label": truth_lookup.get(filename), "raw": text})
        except Exception as e:
            results.append({"filename": filename, "predicted": None,
                            "true_label": truth_lookup.get(filename), "raw": f"ERROR: {e}"})
        if (i + 1) % 20 == 0:
            print(f"  {name}: {i+1}/{len(subset)}")
    c, a, w, t = score_results(results)
    print(f"\n[{name}] {model}: correct {c}/{t} ({c/t*100:.1f}%)  abstain {a}  wrong {w}")
    print("  predicted:", dict(Counter(r['predicted'] for r in results).most_common(6)))
    with open(f"exp_{name}.json", "w") as f:
        json.dump(results, f, indent=2)
    return results

baseline = [r for r in results_stage2_v2 if r['filename'] in dev_files]
c, a, w, t = score_results(baseline)
print(f"\n[baseline v2, haiku] correct {c}/{t} ({c/t*100:.1f}%)  abstain {a}  wrong {w}")

In [ ]:
def enhanced_crop(image_path, bbox, min_short_edge=512):
    img = crop_detection(image_path, bbox, padding=0.25).convert("RGB")
    img = ImageOps.autocontrast(img, cutoff=1)
    w, h = img.size
    s = min_short_edge / min(w, h)
    if s > 1:
        img = img.resize((int(w * s), int(h * s)), Image.LANCZOS)
    return img

def full_frame_with_box(image_path, bbox, max_w=800):
    img = Image.open(image_path).convert("RGB")
    W, H = img.size
    x, y, w, h = bbox
    ImageDraw.Draw(img).rectangle([x*W, y*H, (x+w)*W, (y+h)*H], outline="red", width=max(3, W // 300))
    s = max_w / W
    if s < 1:
        img = img.resize((int(W * s), int(H * s)), Image.LANCZOS)
    return img

FEATURES = """Distinguishing features to check:
- bird: beak, thin legs, feather texture, rounded compact body, fanned or pointed tail
- hedgehog: spiny/bristly dorsal texture, short legs, no visible tail
- possum: long furry tail, large rounded ears, bright eye-shine
- cat: long slender tail, upright pointed ears, elongated body
- rat / ship rat / norway rat / mouse: long thin hairless tail, pointed snout
- rabbit / hare: long ears, powerful hind legs
- mustelid: long low sinuous body, short legs"""

def make_prompt(legend):
    return f"""These are camera-trap images.
{legend}

{FEATURES}

Use only what is visible. Do not rely on assumptions about which species are common in any region.
Choose the single most likely category from this exact list (exact spelling):
{category_list_str}

Respond in exactly this format:
OBSERVATION: <2-3 objective visual details you actually see>
CATEGORY: <one category from the list>
NOTE: <one short sentence>
If you genuinely cannot tell, use CATEGORY: unclassifiable."""

PROMPT_CROP    = make_prompt("Image 1: a close crop of the detected animal.")
PROMPT_CONTEXT = make_prompt("Image 1: the full camera-trap frame with the detection boxed in red (use it for scale and habitat). Image 2: a close crop of the boxed region.")

In [ ]:
def content_crop_only(filename, bbox):
    img = crop_detection(f"sample_images/{filename}", bbox)
    return [img_block(img), {"type": "text", "text": PROMPT_CROP}]

exp_A = run_experiment("A_sonnet_crop", content_crop_only, "claude-sonnet-5")

In [ ]:
def content_context(filename, bbox):
    path = f"sample_images/{filename}"
    return [img_block(full_frame_with_box(path, bbox)),
            img_block(enhanced_crop(path, bbox)),
            {"type": "text", "text": PROMPT_CONTEXT}]

exp_B = run_experiment("B_sonnet_context", content_context, "claude-sonnet-5")

In [ ]:
for r in exp_A[:3]:
    print("---", r['filename'], "| true:", r['true_label'], "| parsed:", r['predicted'])
    print(r['raw'][:400])

In [ ]:
import os, urllib.request
from collections import defaultdict

file_to_entry = {e['file_name']: e for e in coco_data['images']}
seq_to_entries = defaultdict(list)
for e in coco_data['images']:
    seq_to_entries[e['seq_id']].append(e)

os.makedirs("burst_images", exist_ok=True)

def sibling_files(filename):
    entry = file_to_entry.get(filename)
    if not entry:
        return []
    sibs = sorted(seq_to_entries[entry['seq_id']], key=lambda e: e.get('frame_num', 0))
    return [e['file_name'] for e in sibs if e['file_name'] != filename]

def ensure_downloaded(fname):
    local = f"burst_images/{fname}"
    if not os.path.exists(local):
        urllib.request.urlretrieve(base_url + fname, local)
    return local

needed = set()
for f, _ in dev_set:
    needed.update(sibling_files(f))
print(f"Downloading {len(needed)} sibling frames...")
for i, fname in enumerate(needed):
    try:
        ensure_downloaded(fname)
    except Exception as e:
        print("  failed:", fname, e)
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(needed)}")
print("Done.")

In [ ]:
def content_burst(filename, bbox):
    path = f"sample_images/{filename}"
    sibs = sibling_files(filename)
    blocks = [img_block(full_frame_with_box(path, bbox)), img_block(enhanced_crop(path, bbox))]
    for s in sibs:
        local = f"burst_images/{s}"
        if os.path.exists(local):
            blocks.append(img_block(full_frame_with_box(local, bbox, max_w=700)))
    legend = ("Image 1: the full frame with the detection boxed in red. Image 2: a close crop of the boxed region. "
              "Images 3+: the other frames from the same 3-shot burst, same box drawn — the animal may be clearer "
              "or in a different position in these.")
    blocks.append({"type": "text", "text": make_prompt(legend)})
    return blocks

exp_C = run_experiment("C_sonnet_burst", content_burst, "claude-sonnet-5")

In [ ]:
needed_full = set()
for f, _ in confident_images:
    needed_full.update(sibling_files(f))

new_downloads = needed_full - {f for f in needed_full if os.path.exists(f"burst_images/{f}")}
print(f"{len(new_downloads)} new sibling frames to fetch (rest already downloaded)")

for i, fname in enumerate(new_downloads):
    try:
        ensure_downloaded(fname)
    except Exception as e:
        print("  failed:", fname, e)
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(new_downloads)}")
print("Done.")

In [ ]:
results_final = run_experiment("C_full_174", content_burst, "claude-sonnet-5", subset=confident_images)

In [ ]:
import json
with open("results_final_174.json", "w") as f:
    json.dump(results_final, f, indent=2)
print("Saved.")

In [ ]:
from google.colab import files
files.download("results_final_174.json")

In [ ]:
bird_images = [r for r in results_final if r['true_label'] == 'bird']
bird_correct = sum(1 for r in bird_images if r['predicted'] == 'bird')
bird_abstained = sum(1 for r in bird_images if r['predicted'] == 'unclassifiable')
bird_wrong = len(bird_images) - bird_correct - bird_abstained
print(f"True birds: {len(bird_images)}")
print(f"  Correctly identified: {bird_correct}")
print(f"  Abstained: {bird_abstained}")
print(f"  Misclassified as something else: {bird_wrong}")

In [ ]:
import random

# --- Stage 1: blank-filtering stats, recomputed fresh ---
s1_tp = s1_tn = s1_fp = s1_fn = 0
for img in detection_results['images']:
    true_label = truth_lookup.get(img['file'])
    if true_label is None:
        continue
    confident_dets = [d for d in img['detections'] if float(d['conf']) >= CONF_THRESHOLD]
    md_says_animal = len(confident_dets) > 0
    truth_says_animal = (true_label != 'empty')
    if md_says_animal and truth_says_animal: s1_tp += 1
    elif not md_says_animal and not truth_says_animal: s1_tn += 1
    elif md_says_animal and not truth_says_animal: s1_fp += 1
    else: s1_fn += 1
s1_total = s1_tp + s1_tn + s1_fp + s1_fn
s1_accuracy = (s1_tp + s1_tn) / s1_total * 100

# --- Stage 2: species-ID comparison across every version you tested ---
def stats(results):
    t = len(results)
    c = sum(1 for r in results if r['predicted'] == r['true_label'])
    a = sum(1 for r in results if r['predicted'] == 'unclassifiable')
    w = t - c - a
    committed = c / (c + w) * 100 if (c + w) else 0
    return {"total": t, "correct": c, "abstain": a, "wrong": w, "raw_pct": c/t*100, "committed_pct": committed}

comparison = [
    ("Haiku, with location hint (174 images)", stats(results_stage2)),
    ("Haiku, location hint removed (174 images)", stats(results_stage2_v2)),
    ("Sonnet, crop only — Exp A (80-image dev set)", stats(exp_A)),
    ("Sonnet, + full frame — Exp B (80-image dev set)", stats(exp_B)),
    ("Sonnet, + burst frames — Exp C (80-image dev set)", stats(exp_C)),
    ("Sonnet, + burst frames — FINAL (all 174 images)", stats(results_final)),
]

# --- Gallery: a fair mix, not cherry-picked ---
random.seed(42)
full_bbox_lookup = dict(confident_images)
correct_r  = [r for r in results_final if r['predicted'] == r['true_label']]
wrong_r    = [r for r in results_final if r['predicted'] not in (r['true_label'], 'unclassifiable')]
abstain_r  = [r for r in results_final if r['predicted'] == 'unclassifiable']

gallery = (random.sample(correct_r, min(20, len(correct_r))) +
           random.sample(wrong_r, min(10, len(wrong_r))) +
           random.sample(abstain_r, min(8, len(abstain_r))))
random.shuffle(gallery)
print(f"Gallery: {len(gallery)} images — {len(correct_r)} correct / {len(wrong_r)} wrong / {len(abstain_r)} abstained available to sample from")

In [ ]:
def card_html(r):
    bbox = full_bbox_lookup.get(r['filename'])
    try:
        img_data = to_b64(enhanced_crop(f"sample_images/{r['filename']}", bbox, min_short_edge=280))
    except Exception:
        img_data = ""
    predicted, true_label = r['predicted'] or "error", r['true_label']
    if predicted == true_label:
        cls, label = "correct", "✓ Correct"
    elif predicted == "unclassifiable":
        cls, label = "abstain", "— Abstained"
    else:
        cls, label = "wrong", "✗ Wrong"
    return f'''<div class="card {cls}">
<img src="data:image/jpeg;base64,{img_data}">
<div class="body"><span class="tag">{label}</span>
<p>Predicted: <b>{predicted}</b><br>True label: <b>{true_label}</b></p></div></div>'''

rows = "\n".join(
    f"<tr><td>{name}</td><td>{s['total']}</td><td>{s['correct']}</td><td>{s['abstain']}</td>"
    f"<td>{s['wrong']}</td><td>{s['raw_pct']:.1f}%</td><td>{s['committed_pct']:.1f}%</td></tr>"
    for name, s in comparison
)
cards = "\n".join(card_html(r) for r in gallery)

html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>AI Camera-Trap Pipeline — Results</title>
<style>
body {{ font-family: -apple-system, Arial, sans-serif; max-width: 1000px; margin: 40px auto; padding: 0 20px; color: #1a1a1a; }}
h1 {{ font-size: 26px; }} h2 {{ margin-top: 40px; border-bottom: 2px solid #eee; padding-bottom: 6px; }}
.stat-row {{ display: flex; gap: 20px; margin: 20px 0; }}
.stat {{ background: #f6f6f6; border-radius: 10px; padding: 16px 20px; flex: 1; }}
.stat .num {{ font-size: 28px; font-weight: 700; }} .stat .lbl {{ color: #666; font-size: 13px; }}
table {{ width: 100%; border-collapse: collapse; font-size: 14px; }}
th, td {{ text-align: left; padding: 8px 10px; border-bottom: 1px solid #eee; }}
th {{ background: #fafafa; }}
.gallery {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(220px, 1fr)); gap: 16px; margin-top: 20px; }}
.card {{ border-radius: 10px; overflow: hidden; border: 2px solid #eee; }}
.card.correct {{ border-color: #2e7d32; }} .card.wrong {{ border-color: #c62828; }} .card.abstain {{ border-color: #999; }}
.card img {{ width: 100%; height: 160px; object-fit: cover; display: block; }}
.card .body {{ padding: 10px; font-size: 13px; }}
.tag {{ font-weight: 600; }}
</style></head><body>

<h1>AI-Powered Camera-Trap Pipeline</h1>
<p>MegaDetector + Claude vision, tested on 200 images from the Wellington Camera Traps dataset (LILA BC).</p>

<h2>Stage 1 — Blank-frame filtering (MegaDetector)</h2>
<div class="stat-row">
<div class="stat"><div class="num">{s1_accuracy:.1f}%</div><div class="lbl">Overall accuracy</div></div>
<div class="stat"><div class="num">{s1_tp}/{s1_tp+s1_fn}</div><div class="lbl">True animals correctly kept</div></div>
<div class="stat"><div class="num">{s1_fn}</div><div class="lbl">Animals wrongly filtered out</div></div>
</div>

<h2>Stage 2 — Species identification (Claude)</h2>
<p>Progression from the baseline through each tested improvement — full data behind every number in the notebook.</p>
<table><tr><th>Configuration</th><th>N</th><th>Correct</th><th>Abstained</th><th>Wrong</th><th>Raw accuracy</th><th>Accuracy when committed</th></tr>
{rows}</table>

<h2>Sample results ({len(gallery)} of {len(results_final)}, mixed correct / wrong / abstained)</h2>
<div class="gallery">{cards}</div>

</body></html>"""

with open("camera_trap_report.html", "w") as f:
    f.write(html)
print(f"Saved — {len(html)/1024:.0f} KB")

from google.colab import files
files.download("camera_trap_report.html")

In [ ]:
import pandas as pd

gt_df = pd.read_csv("sample_ground_truth.csv")
example_per_label = gt_df.groupby("true_label").first().reset_index()

def cheat_thumb(filename, min_short_edge=220):
    bbox = full_bbox_lookup.get(filename)
    if bbox:
        img = enhanced_crop(f"sample_images/{filename}", bbox, min_short_edge=min_short_edge)
    else:
        img = Image.open(f"sample_images/{filename}").convert("RGB")
        img.thumbnail((min_short_edge*2, min_short_edge*2))
    return to_b64(img)

cheat_cards = "".join(
    f'<div class="ccard"><img src="data:image/jpeg;base64,{cheat_thumb(row.filename)}">'
    f'<div class="clabel">{row.true_label}</div></div>\n'
    for row in example_per_label.itertuples()
)

cheat_html = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Species Cheat Sheet</title>
<style>
body {{ font-family: -apple-system, Arial, sans-serif; max-width: 900px; margin: 30px auto; }}
.grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 14px; }}
.ccard {{ border: 1px solid #ddd; border-radius: 8px; overflow: hidden; }}
.ccard img {{ width: 100%; height: 120px; object-fit: cover; display: block; }}
.clabel {{ text-align: center; padding: 6px; font-weight: 600; text-transform: capitalize; }}
</style></head><body>
<h1>Species reference</h1>
<p>One example per category present in this sample — for quick reference during manual review.</p>
<div class="grid">{cheat_cards}</div></body></html>"""

with open("species_cheat_sheet.html", "w") as f:
    f.write(cheat_html)
print(f"Saved — {len(example_per_label)} categories covered")

from google.colab import files
files.download("species_cheat_sheet.html")

In [ ]:
import random

random.seed(99)
timing_sample = random.sample(gt_df["filename"].tolist(), 30)
stage2_lookup = {r['filename']: r for r in results_final}

def pipeline_verdict(filename):
    if filename in stage2_lookup:
        r = stage2_lookup[filename]
        return r['predicted'], (r.get('raw', '') or '')[:200]
    return 'BLANK (filtered by MegaDetector)', 'Confidence below threshold — no animal detected.'

def page(filename_list, title, instructions, show_verdict):
    cards = ""
    for i, fn in enumerate(filename_list, 1):
        bbox = full_bbox_lookup.get(fn)
        img = enhanced_crop(f"sample_images/{fn}", bbox, 400) if (show_verdict and bbox) else Image.open(f"sample_images/{fn}").convert("RGB")
        if not (show_verdict and bbox):
            img.thumbnail((500, 500))
        extra = ""
        if show_verdict:
            verdict, note = pipeline_verdict(fn)
            extra = f'<p><b>Pipeline says:</b> {verdict}<br><span style="color:#666;font-size:13px">{note}</span></p>'
        cards += f'<div style="margin-bottom:30px"><div style="font-weight:700">#{i}</div><img src="data:image/jpeg;base64,{to_b64(img)}" style="max-width:100%;border-radius:8px;border:1px solid #ddd">{extra}</div>\n'
    return f'<!DOCTYPE html><html><head><meta charset="utf-8"><title>{title}</title></head><body style="font-family:-apple-system,Arial,sans-serif;max-width:700px;margin:30px auto"><h1>{title}</h1><p>{instructions}</p>{cards}</body></html>'

with open("manual_review_batch.html", "w") as f:
    f.write(page(timing_sample, "Manual review — 30 images",
                  "Start a stopwatch. Scroll through and privately decide, for each: blank or animal, and if an animal, which species (use the cheat sheet). Don't write anything down — just note your total time at the end.",
                  show_verdict=False))

with open("tool_review_batch.html", "w") as f:
    f.write(page(timing_sample, "Reviewing the pipeline's output — same 30 images",
                  "Start a stopwatch. Scroll through and just sanity-check each verdict. Note your total time at the end.",
                  show_verdict=True))

print("Saved both.")
from google.colab import files
files.download("manual_review_batch.html")
files.download("tool_review_batch.html")

In [ ]:
# Pick the clearest example per species, using detection confidence as a proxy for image quality
detection_conf_lookup = {}
for img in detection_results['images']:
    confident_dets = [d for d in img['detections'] if float(d['conf']) >= CONF_THRESHOLD]
    if confident_dets:
        detection_conf_lookup[img['file']] = max(float(d['conf']) for d in confident_dets)

gt_df['det_conf'] = gt_df['filename'].map(detection_conf_lookup).fillna(0)
example_per_label = gt_df.sort_values('det_conf', ascending=False).groupby('true_label').first().reset_index()

# Rebuild the cheat sheet with the better examples
cheat_cards = "".join(
    f'<div class="ccard"><img src="data:image/jpeg;base64,{cheat_thumb(row.filename)}">'
    f'<div class="clabel">{row.true_label}</div></div>\n'
    for row in example_per_label.itertuples()
)

cheat_html = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Species Cheat Sheet</title>
<style>
body {{ font-family: -apple-system, Arial, sans-serif; max-width: 900px; margin: 30px auto; }}
.grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 14px; }}
.ccard {{ border: 1px solid #ddd; border-radius: 8px; overflow: hidden; }}
.ccard img {{ width: 100%; height: 120px; object-fit: cover; display: block; }}
.clabel {{ text-align: center; padding: 6px; font-weight: 600; text-transform: capitalize; }}
</style></head><body>
<h1>Species reference</h1>
<p>One example per category present in this sample — for quick reference during manual review.</p>
<div class="grid">{cheat_cards}</div></body></html>"""

with open("species_cheat_sheet.html", "w") as f:
    f.write(cheat_html)
print(f"Saved — {len(example_per_label)} categories covered")

from google.colab import files
files.download("species_cheat_sheet.html")

In [ ]:
missing_sibs = set()
for fn in timing_sample:
    for s in sibling_files(fn):
        if not os.path.exists(f"burst_images/{s}"):
            missing_sibs.add(s)
print(f"Downloading {len(missing_sibs)} more sibling frames for the timing sample...")
for fname in missing_sibs:
    try:
        ensure_downloaded(fname)
    except Exception as e:
        print("  failed:", fname, e)
print("Done.")

def burst_card_images(fn):
    bbox = full_bbox_lookup.get(fn)
    def load(path):
        return enhanced_crop(path, bbox, 350) if bbox else (lambda im: (im.thumbnail((450,450)), im)[1])(Image.open(path).convert("RGB"))
    imgs = [load(f"sample_images/{fn}")]
    for s in sibling_files(fn):
        local = f"burst_images/{s}"
        if os.path.exists(local):
            imgs.append(load(local))
    return imgs

def page_burst(filename_list, title, instructions, show_verdict):
    cards = ""
    for i, fn in enumerate(filename_list, 1):
        imgs = burst_card_images(fn)
        img_tags = "".join(f'<img src="data:image/jpeg;base64,{to_b64(im)}" style="max-width:32%;border-radius:6px;border:1px solid #ddd;margin-right:4px">' for im in imgs)
        extra = ""
        if show_verdict:
            verdict, note = pipeline_verdict(fn)
            extra = f'<p><b>Pipeline says:</b> {verdict}<br><span style="color:#666;font-size:13px">{note}</span></p>'
        cards += f'<div style="margin-bottom:30px"><div style="font-weight:700">#{i} ({len(imgs)} frame{"s" if len(imgs)>1 else ""})</div><div style="display:flex;flex-wrap:wrap">{img_tags}</div>{extra}</div>\n'
    return f'<!DOCTYPE html><html><head><meta charset="utf-8"><title>{title}</title></head><body style="font-family:-apple-system,Arial,sans-serif;max-width:750px;margin:30px auto"><h1>{title}</h1><p>{instructions}</p>{cards}</body></html>'

with open("manual_review_batch.html", "w") as f:
    f.write(page_burst(timing_sample, "Manual review — 30 images (all burst frames shown)",
        "Start a stopwatch. For each numbered group you're shown every available frame from that camera trigger — look across all of them. Privately decide: blank or animal, and if an animal, which species. Note your total time at the end.", False))

with open("tool_review_batch.html", "w") as f:
    f.write(page_burst(timing_sample, "Reviewing the pipeline's output — same 30 images",
        "Start a stopwatch. Sanity-check each verdict against the frame(s) shown. Note your total time.", True))

print("Saved both, with burst frames.")
from google.colab import files
files.download("manual_review_batch.html")
files.download("tool_review_batch.html")

In [ ]:
print("Your 30-image timing sample, where the pipeline's verdict didn't match ground truth:\n")
for fn in timing_sample:
    r = stage2_lookup.get(fn)
    if r is None:
        continue  # was filtered as blank by stage 1
    if r['predicted'] != r['true_label']:
        print(f"{fn}")
        print(f"  True label: {r['true_label']}   Pipeline said: {r['predicted']}")
        print(f"  Claude's reasoning: {(r.get('raw','') or '')[:200]}")
        print()

In [ ]:
r = stage2_lookup['190916094406017b5952.JPG']
print(r['raw'])